# N076 · 树的原地变换与Morris遍历

**目标：** 在修改结构时保留并最终恢复必要的链接。

**先修：** N046, N069, N074。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 展开链表；镜像；临时线索；Morris；恢复义务；原地空间。

## 从问题到算法

原地树变换的验收应检查指针结构。展开树要求右链是原前序，所有left必须清空。Morris遍历临时把中序前驱的空right指向当前节点，借这条线索返回祖先；第二次遇到线索时立即拆除，遍历结束必须完全恢复原树。不能为了提前拿到第k项直接中断而遗留线程。

## 最小实现

**本章接口：** `flatten(root)`、`invert_tree(root)`、`morris_inorder(root)`

In [1]:
class TreeNode:
    def __init__(self,val=0,left=None,right=None):
        self.val,self.left,self.right=val,left,right

def tree_from_level(values):
    """Construct a finite binary tree from level order with None placeholders."""
    from collections import deque
    if not values or values[0] is None: return None
    root=TreeNode(values[0]); queue=deque([root]); i=1
    while queue and i<len(values):
        node=queue.popleft()
        if values[i] is not None: node.left=TreeNode(values[i]); queue.append(node.left)
        i+=1
        if i<len(values):
            if values[i] is not None: node.right=TreeNode(values[i]); queue.append(node.right)
            i+=1
    return root

def flatten(root):
    stack=[root] if root else []; previous=None
    while stack:
        node=stack.pop()
        if node.right: stack.append(node.right)
        if node.left: stack.append(node.left)
        if previous: previous.left=None; previous.right=node
        previous=node
    if previous: previous.left=None; previous.right=None

def invert_tree(root):
    stack=[root] if root else []
    while stack:
        node=stack.pop(); node.left,node.right=node.right,node.left
        if node.left: stack.append(node.left)
        if node.right: stack.append(node.right)
    return root

def morris_inorder(root):
    answer=[]; current=root
    while current:
        if current.left is None: answer.append(current.val); current=current.right
        else:
            predecessor=current.left
            while predecessor.right and predecessor.right is not current: predecessor=predecessor.right
            if predecessor.right is None: predecessor.right=current; current=current.left
            else: predecessor.right=None; answer.append(current.val); current=current.right
    return answer

## 正确性、前提与复杂度

展开时先保存原孩子再修改前一节点，不会失去待访问子树。Morris每个临时线索只建立一次、拆除一次，结束时原本为空的前驱right恢复为空。

**代价：** 均O(n)时间；本章flatten/invert显式栈O(h)，Morris辅助空间O(1)，输出O(n)。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

r=tree_from_level([1,2,5,3,4,None,6]); before=morris_inorder(r); flatten(r)
chain=[]; n=r
while n: chain.append(n.val); n=n.right
show_table(['对象','次序'],[('原中序',before),('展开后前序右链',chain)])

对象,次序
原中序,"[3, 2, 4, 1, 5, 6]"
展开后前序右链,"[1, 2, 3, 4, 5, 6]"


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
r=tree_from_level([1,2,5,3,4,None,6]); snapshot={}; preorder=[]; stack=[r]
while stack:
    n=stack.pop(); snapshot[n]=(n.left,n.right); preorder.append(n)
    if n.right: stack.append(n.right)
    if n.left: stack.append(n.left)
assert morris_inorder(r)==[3,2,4,1,5,6]
assert all((n.left,n.right)==edges for n,edges in snapshot.items())
invert_tree(invert_tree(r)); assert all((n.left,n.right)==edges for n,edges in snapshot.items())
flatten(r); after=[]; n=r
while n:
    assert n.left is None and n not in after; after.append(n); n=n.right
assert after==preorder
print('N076: 所有本章断言通过')

N076: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 证明Morris临时边最后会删除。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 解释早停遍历可能破坏恢复。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 114. Flatten Binary Tree to Linked List（canonical）
- 226. Invert Binary Tree（canonical）
- 94. Binary Tree Inorder Traversal（extension）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。